# World Cup 2026 prediction model — exploration

Run after `wc26 ingest`, `wc26 features`, `wc26 train`. Covers: Elo trajectories, feature importance, validation calibration, and the simulated remaining bracket.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wc26.config import PROCESSED_DIR
from wc26.features.build import FeatureBuilder, feature_columns

matches = pd.read_parquet(PROCESSED_DIR / 'matches.parquet')
features = pd.read_parquet(PROCESSED_DIR / 'features.parquet')
fb = FeatureBuilder(matches)
print(len(matches), 'matches |', len(features), 'feature rows')

## Elo trajectories of the 2026 semifinalists

In [ ]:
dates = pd.date_range('2018-01-01', '2026-07-15', freq='MS')
fig, ax = plt.subplots(figsize=(11, 5))
for team in ['Spain', 'France', 'England', 'Argentina', 'Brazil', 'Germany']:
    ax.plot(dates, [fb.elo.rating(team, d) for d in dates], label=team)
ax.legend(); ax.set_title('Elo ratings'); ax.grid(alpha=0.3)
plt.show()

## Feature importance (production model)

In [ ]:
from wc26.model.train import load
model, rho, temp = load('prod')
imp = pd.Series(model.feature_importances_, index=model.feature_names_in_)
imp.nlargest(25).sort_values().plot.barh(figsize=(8, 9), title='XGBoost gain importance (top 25)')
plt.tight_layout(); plt.show()

## Calibration on the 2026 World Cup (played matches)

In [ ]:
from wc26.model.train import match_probs
val = features[(features['is_wc'] == 1) & (features['date'] >= '2026-06-01')]
rows = []
for mid, pair in val.groupby('match_id'):
    pair = pair.sort_values('side')
    if len(pair) != 2:
        continue
    r = match_probs(model, pair, rho, temp)
    gf, ga = pair.iloc[0]['goals'], pair.iloc[0]['goals_against']
    rows.append({'p_win': r['p_win'], 'won': float(gf > ga),
                 'p_draw': r['p_draw'], 'drew': float(gf == ga),
                 'team': pair.iloc[0]['team'], 'opp': pair.iloc[0]['opponent'],
                 'score': f"{int(gf)}-{int(ga)}"})
cal = pd.DataFrame(rows)
bins = pd.cut(cal['p_win'], np.arange(0, 1.05, 0.2))
print(cal.groupby(bins, observed=True).agg(pred=('p_win', 'mean'), actual=('won', 'mean'), n=('won', 'size')))
cal['surprise'] = -np.log(np.where(cal['won'] == 1, cal['p_win'], np.where(cal['drew'] == 1, cal['p_draw'], 1 - cal['p_win'] - cal['p_draw'])))
print('\nBiggest surprises of the tournament:')
print(cal.nlargest(8, 'surprise')[['team', 'opp', 'score', 'p_win', 'surprise']])

## Simulated remaining bracket

In [ ]:
from wc26.model.pipeline import load_predictor
from wc26.sim.monte_carlo import simulate_remaining
pred = load_predictor()
weather = pd.read_parquet(PROCESSED_DIR / 'wc_match_weather.parquet')
out = simulate_remaining(pred, matches, weather, as_of='2026-07-15', n=10000)
champ = pd.Series(out['p_champion'])
champ.sort_values().plot.barh(title='P(World Cup 2026 champion)', figsize=(7, 3))
plt.show()
out['sf2'], out['p_champion']